# Notebook 2b — Concentration Inequalities for SWAP-Test Circuits

**Learning goals**

1. Build a SWAP-test circuit that compares two classical vectors
   via amplitude encoding.
2. Compute the **ideal** ancilla-0 probability `p0_ideal` from
   an exact statevector simulation.
3. Simulate **finite shots** to get `p0_hat`, and use:
      - Hoeffding
      - Chernoff
      - Chebyshev (empirical variance)
   to:
      - plan shot budgets,
      - compute confidence intervals,
      - study errors across many vector pairs.

This notebook is the SWAP-specific counterpart of:
 - Notebook 1 (polls → QKD → observables),
 - Notebook 2 (1-qubit example + NEAT/Runtime template).

Here we focus on **SWAP tests**.

## 0. Setup: Imports & Concentration Inequalities

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(2025)

# --- Concentration inequality helpers (same logic as Notebook 1/2) ---

def hoeffding_bound(p_est: float, n: int, delta: float):
    """Two-sided Hoeffding CI for Bernoulli / [0,1]-bounded variables."""
    if n <= 0:
        raise ValueError("n must be positive")
    radius = math.sqrt(math.log(2 / delta) / (2 * n))
    return max(0.0, p_est - radius), min(1.0, p_est + radius)

def chernoff_bound(p_est: float, n: int, delta: float):
    """Simple Chernoff-style CI: P(|p_hat - p| >= eps) ≤ 2 exp(- n eps^2 / 3)."""
    if n <= 0:
        raise ValueError("n must be positive")
    radius = math.sqrt(3 * math.log(2 / delta) / n)
    return max(0.0, p_est - radius), min(1.0, p_est + radius)

def chebyshev_bound(p_est: float, n: int, delta: float):
    """
    Two-sided Chebyshev CI using *empirical* variance for Bernoulli.
    Not strictly guaranteed, but useful as a contrast.
    """
    if n <= 0:
        raise ValueError("n must be positive")

    var_est = p_est * (1 - p_est)
    if var_est == 0:
        return p_est, p_est

    radius = math.sqrt(var_est / (delta * n))
    return max(0.0, p_est - radius), min(1.0, p_est + radius)

def sample_size_hoeffding(epsilon: float, delta: float) -> int:
    """2 exp(-2 n ε²) ≤ δ  ⇒  n ≥ (1 / (2 ε²)) * log(2/δ)."""
    if epsilon <= 0 or delta <= 0 or delta >= 1:
        raise ValueError("epsilon > 0 and 0 < delta < 1 required.")
    return int(math.ceil((1.0 / (2 * epsilon**2)) * math.log(2.0 / delta)))

def sample_size_chebyshev(epsilon: float, delta: float) -> int:
    """Chebyshev with worst-case variance σ² ≤ 1/4."""
    if epsilon <= 0 or delta <= 0 or delta >= 1:
        raise ValueError("epsilon > 0 and 0 < delta < 1 required.")
    return int(math.ceil(1.0 / (4 * epsilon**2 * delta)))

def sample_size_chernoff(epsilon: float, delta: float) -> int:
    """Chernoff-style bound sample complexity."""
    if epsilon <= 0 or delta <= 0 or delta >= 1:
        raise ValueError("epsilon > 0 and 0 < delta < 1 required.")
    return int(math.ceil((3.0 / (epsilon**2)) * math.log(2.0 / delta)))


## 1. Building a SWAP-Test Circuit from Classical Vectors

We encode two real vectors `v1`, `v2` as quantum states using amplitude
encoding, then run a SWAP test on them.

SWAP-test structure:

- Registers:
    * |ψ⟩ on register A,
    * |φ⟩ on register B,
    * ancilla qubit |0⟩.
- Circuit:
    * H on ancilla,
    * controlled-SWAP (ancilla as control) between A and B,
    * H on ancilla.
- Measurement:
    * measure ancilla in computational basis.

Probability of ancilla=0:

  P(0) = (1 + |⟨ψ|φ⟩|²) / 2

which is a bounded quantity in [1/2, 1]. Perfectly orthogonal states
give P(0) = 1/2, identical states give P(0) = 1.

In [ ]:
from math import log2, ceil
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def build_swap_test_from_vectors(v1_raw, v2_raw):
    """
    Build a SWAP-test circuit from two classical vectors.
    Steps:
    - convert to float np.arrays
    - pad to next power-of-2 dimension
    - normalize
    - amplitude-encode into two registers
    - apply SWAP test (Hadamard, controlled SWAPs, Hadamard)
    """
    v1 = np.asarray(v1_raw, dtype=float)
    v2 = np.asarray(v2_raw, dtype=float)

    if len(v1) != len(v2):
        raise ValueError("v1 and v2 must have same length")

    d = len(v1)
    if d == 0:
        raise ValueError("Zero-dimensional vectors not allowed")

    # Pad to next power of 2
    num_qubits_vec = ceil(log2(d))
    dim = 2**num_qubits_vec
    if d < dim:
        v1 = np.pad(v1, (0, dim - d), constant_values=0.0)
        v2 = np.pad(v2, (0, dim - d), constant_values=0.0)

    # Normalize
    n1 = np.linalg.norm(v1)
    n2 = np.linalg.norm(v2)
    if n1 == 0 or n2 == 0:
        raise ValueError("Zero vector encountered; cannot normalize.")

    v1 = v1 / n1
    v2 = v2 / n2

    # Total qubits: A (n), B (n), ancilla (1)
    total_qubits = 2 * num_qubits_vec + 1
    qc = QuantumCircuit(total_qubits)

    qA = list(range(num_qubits_vec))
    qB = list(range(num_qubits_vec, 2 * num_qubits_vec))
    anc = 2 * num_qubits_vec

    # Initialize amplitude-encoded states
    qc.initialize(v1, qA)
    qc.initialize(v2, qB)

    # SWAP test core
    qc.h(anc)
    for i in range(num_qubits_vec):
        qc.cswap(anc, qA[i], qB[i])
    qc.h(anc)

    return qc, anc


## 1.1 Ideal Ancilla-0 Probability from Statevector

Given a SWAP-test circuit `qc` and ancilla index `anc`, we can:

 - simulate the statevector after the SWAP test,
 - compute P(ancilla=0) by summing probabilities of all basis states
   whose ancilla bit is 0.

This is our **p0_ideal** — the analogue of a NEAT `ideal_sim` value,
but computed here with standard Qiskit Statevector.

In [ ]:
def swap_test_p0_statevector(v1, v2):
    qc, anc = build_swap_test_from_vectors(v1, v2)
    sv = Statevector.from_instruction(qc)
    probs = sv.probabilities()  # length 2^(2n+1)
    n_qubits = qc.num_qubits

    p0 = 0.0
    for idx, p in enumerate(probs):
        # ancilla is qubit 'anc'. In little-endian convention, qubit 0
        # corresponds to LSB. So we check the ancilla bit in 'idx'.
        if ((idx >> anc) & 1) == 0:
            p0 += p
    return float(p0)

# Quick sanity check on simple vectors
v1 = [1.0, 0.0]  # |0>
v2 = [1.0, 0.0]  # |0>, identical → P(0) should be 1
print("P(0) for identical |0>, |0>:", swap_test_p0_statevector(v1, v2))

v1 = [1.0, 0.0]  # |0>
v2 = [0.0, 1.0]  # |1>, orthogonal → P(0) should be ~0.5
print("P(0) for |0>, |1> (orthogonal):", swap_test_p0_statevector(v1, v2))


### 1.2 P(0) and Similarity

For ideal SWAP tests on pure states |ψ⟩, |φ⟩:
```
  P(ancilla=0) = (1 + |⟨ψ|φ⟩|²) / 2
```
This means:

- If the states are identical (|⟨ψ|φ⟩| = 1), then P(0) = 1.
- If the states are orthogonal (|⟨ψ|φ⟩| = 0), then P(0) = 1/2.

In this notebook, P(0) is the **Bernoulli parameter** we care about.
Every shot of the SWAP test on the ancilla yields:
```
  X_i ∈ {0,1},  P(X_i = 1) = P(ancilla=0) = p0.
```
This is exactly the setting where Hoeffding, Chernoff, and Chebyshev apply.


## 2. Shot Budget for SWAP-Test Estimation

We pick a target:

  - `ε = 0.05`  (tolerance for |p_hat - p_true|),
  - `δ = 0.05`  (5% error probability),

and compute shot budgets from different inequalities.


In [ ]:

epsilon = 0.05
delta = 0.05

m_hoeff = sample_size_hoeffding(epsilon, delta)
m_chern = sample_size_chernoff(epsilon, delta)
m_cheb  = sample_size_chebyshev(epsilon, delta)

print(f"Shot budgets for ε={epsilon}, δ={delta}:")
print(f"  Hoeffding : m = {m_hoeff}")
print(f"  Chernoff  : m = {m_chern}")
print(f"  Chebyshev : m = {m_cheb}")


## 3. Finite-Shot SWAP Experiments on Multiple Vector Pairs

We now:

1. Fix ε, δ and use **Hoeffding** to get a shot budget m_hoeff.
2. For K random vector pairs (v1, v2):
    - build SWAP-test circuit,
    - compute p0_ideal via statevector,
    - simulate m_hoeff shots (Bernoulli draws with parameter p0_ideal),
    - compute p0_hat, error = |p0_hat - p0_ideal|.
3. Count how many cases have error > ε (violations against our target).

In [ ]:

def simulate_swap_shots_p0(v1, v2, shots, rng):
    p0_true = swap_test_p0_statevector(v1, v2)
    # simulate Bernoulli with p0_true, instead of running a full qasm_sim
    counts_0 = rng.binomial(n=shots, p=p0_true)
    return p0_true, counts_0 / shots

K = 30         # number of random vector pairs
dim = 4        # dimension of classical vectors before padding

results = []
for idx in range(1, K + 1):
    v1 = rng.normal(size=dim)
    v2 = rng.normal(size=dim)

    p0_ideal, p0_hat = simulate_swap_shots_p0(v1, v2, m_hoeff, rng)
    err = abs(p0_hat - p0_ideal)
    results.append((idx, p0_ideal, p0_hat, err))

print(f"Using Hoeffding shot budget m = {m_hoeff} for ε = {epsilon}, δ = {delta}")
for idx, p0_ideal, p0_hat, err in results:
    print(f"Pair {idx:02d}: p0_ideal={p0_ideal:.4f}, p0_hat={p0_hat:.4f}, |err|={err:.4f}")

violations = sum(1 for _,_,_,err in results if err > epsilon)

print("\n=== Summary ===")
print(f"Total pairs              : {K}")
print(f"Violations (> ε)         : {violations}")
print(f"Empirical violation rate : {violations / K:.3f} (target δ = {delta})")
print(f"Max abs error            : {max(err for *_, err in results):.4f}")


### 3.1 Visualizing SWAP Errors vs Hoeffding Radius

We compute the Hoeffding radius:

  r_H = sqrt(log(2/δ) / (2m))

and compare each |p0_hat - p0_ideal| against r_H and ε.

In [ ]:
hoeff_radius = math.sqrt(math.log(2 / delta) / (2 * m_hoeff))

pair_ids = [idx for idx,_,_,_ in results]
errors   = [err for *_, err in results]

plt.bar(pair_ids, errors)
plt.axhline(hoeff_radius, linestyle="--", label=f"Hoeffding radius ≈ {hoeff_radius:.3f}")
plt.axhline(epsilon, linestyle=":", label=f"ε = {epsilon}")
plt.xlabel("Pair index")
plt.ylabel("|p0_hat - p0_ideal|")
plt.title("SWAP-test errors vs Hoeffding radius")
plt.legend()
plt.show()


## 4. CI Width and Coverage for SWAP-Test p0

We now study:
  - how wide the CIs are on average,
  - how often they contain the **true** p0_ideal,

for Hoeffding, Chernoff, and Chebyshev, across many random SWAP-test
vector pairs.

In [ ]:

def ci_coverage_and_width_swap(K=50, dim=4, shots=1000, delta=0.05):
    cover_counts = {"Hoeffding": 0, "Chernoff": 0, "Chebyshev": 0}
    width_sums   = {"Hoeffding": 0.0, "Chernoff": 0.0, "Chebyshev": 0.0}

    for _ in range(K):
        v1 = rng.normal(size=dim)
        v2 = rng.normal(size=dim)

        p_true = swap_test_p0_statevector(v1, v2)
        counts_0 = rng.binomial(n=shots, p=p_true)
        p_hat = counts_0 / shots

        # Hoeffding
        lo_h, hi_h = hoeffding_bound(p_hat, shots, delta)
        if lo_h <= p_true <= hi_h:
            cover_counts["Hoeffding"] += 1
        width_sums["Hoeffding"] += (hi_h - lo_h)

        # Chernoff
        lo_c, hi_c = chernoff_bound(p_hat, shots, delta)
        if lo_c <= p_true <= hi_c:
            cover_counts["Chernoff"] += 1
        width_sums["Chernoff"] += (hi_c - lo_c)

        # Chebyshev
        lo_ch, hi_ch = chebyshev_bound(p_hat, shots, delta)
        if lo_ch <= p_true <= hi_ch:
            cover_counts["Chebyshev"] += 1
        width_sums["Chebyshev"] += (hi_ch - lo_ch)

    results = {}
    for name in cover_counts:
        coverage = cover_counts[name] / K
        avg_width = width_sums[name] / K
        results[name] = (coverage, avg_width)
    return results

shots_ci = m_hoeff  # use the same Hoeffding-based budget
K_ci = 100

swap_ci_results = ci_coverage_and_width_swap(K=K_ci, dim=dim, shots=shots_ci, delta=delta)
print(f"SWAP CI experiment: K={K_ci} pairs, shots={shots_ci}, δ={delta}\n")
print("Method       Coverage   Avg width")
print("---------------------------------")
for name, (cov, width) in swap_ci_results.items():
    print(f"{name:<10} {cov:8.3f}   {width:9.4f}")



### 4.1 Visualizing SWAP CI Width & Coverage


In [ ]:

methods   = list(swap_ci_results.keys())
coverages = [swap_ci_results[m][0] for m in methods]
widths    = [swap_ci_results[m][1] for m in methods]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: average CI width
axes[0].bar(methods, widths)
axes[0].set_ylabel("Average CI width")
axes[0].set_title(f"SWAP CI width (shots={shots_ci}, δ={delta})")

# Right: empirical coverage
axes[1].bar(methods, coverages)
axes[1].axhline(1 - delta, linestyle="--", label=f"Target 1 - δ = {1 - delta:.2f}")
axes[1].set_ylabel("Empirical coverage")
axes[1].set_ylim(0, 1.05)
axes[1].set_title(f"SWAP CI coverage vs target (δ = {delta})")
axes[1].legend()

plt.tight_layout()
plt.show()


## 5. SWAP + NEAT + EstimatorV2: Conceptual Template

In this notebook we:
  - built SWAP-test circuits from classical vectors,
  - computed p0_ideal via Statevector (exact),
  - simulated finite shots via Bernoulli sampling,
  - applied Hoeffding/Chernoff/Chebyshev to:
      * plan shot budgets,
      * compute CI widths,
      * check coverage.

In your **full QAMP / thesis setup**, you can replace the
`Statevector`-based p0_ideal with:

  - `Neat.ideal_sim(pubs, cliffordize=True)` for SWAP PUBs,

and the Bernoulli sampling step with:

  - actual Qiskit Runtime `EstimatorV2` runs on a backend.

The pattern is the same:

  1. Build SWAP-test circuit (no measurement) + observable (ancilla Z).
  2. NEAT ideal_sim → <Z>_ideal  →  p0_ideal = (1 + <Z>_ideal) / 2.
  3. Estimator with m shots      →  <Z>_hat   →  p0_hat   = (1 + <Z>_hat) / 2.
  4. Use the same concentration inequalities around p0_hat.

Below is a **template** for doing this in a Runtime environment.


In [ ]:

"""
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator
from qiskit_ibm_runtime.debug_tools import Neat

# 1) Service and backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
estimator = Estimator(mode=backend)
neat = Neat(backend)

# 2) Build SWAP-test PUBs from many vector pairs
pubs = []
for k in range(K):  # e.g. K=30
    v1 = rng.normal(size=dim)
    v2 = rng.normal(size=dim)
    qc, anc = build_swap_test_from_vectors(v1, v2)

    n_qubits = qc.num_qubits
    paulis = ["I"] * n_qubits
    # In Pauli strings, rightmost char corresponds to qubit 0 (little-endian)
    paulis[n_qubits - 1 - anc] = "Z"
    pauli_str = "".join(paulis)
    observable = SparsePauliOp(pauli_str)

    pubs.append((qc, [observable], [[]]))  # no parameters

# 3) NEAT ideal_sim for <Z>_ideal
neat_res = neat.ideal_sim(pubs, cliffordize=True)
z_ideals = [pr.data.evs[0] for pr in neat_res]  # one observable per PUB
p0_ideals = [(z + 1.0) / 2.0 for z in z_ideals]

# 4) Estimator finite-shot runs with m shots (Hoeffding)
epsilon = 0.05
delta = 0.05
m_shots = sample_size_hoeffding(epsilon, delta)

job = estimator.run(pubs, options={"shots": m_shots})
est_res = job.result()
z_hats = [pr.data.evs[0] for pr in est_res]
p0_hats = [(z + 1.0) / 2.0 for z in z_hats]

# 5) For each pair, compute error and CIs
errors = []
violations = 0
for p0_true, p0_hat in zip(p0_ideals, p0_hats):
    err = abs(p0_hat - p0_true)
    errors.append(err)
    if err > epsilon:
        violations += 1

print(f"Violations (> ε): {violations} / {len(p0_ideals)}")

# Similarly, you can compute CI widths & coverage exactly as we did
# in this notebook, but now p0_true comes from NEAT ideal_sim
# and p0_hat from Estimator on real/noisy hardware.
"""


## 6. Wrap-Up

In this SWAP-focused notebook, we:

- Constructed SWAP-test circuits from classical vectors.
- Computed ideal ancilla-0 probabilities `p0_ideal` via Statevector.
- Simulated finite-shot estimates `p0_hat` and quantified errors.
- Used Hoeffding/Chernoff/Chebyshev:
    * to derive shot budgets m(ε, δ),
    * to compute CI widths,
    * to check empirical coverage.